In [9]:
# MSCS & MSDS OOP with Python Assignment 2
# Mini-Project 1: Fetching and Displaying a Weather Forecast

import os
import json
import time
import urllib.parse
import urllib.request
import requests

# CONFIGURATION

LOCATION = "Mpigi"
CONTACT_EMAIL = os.getenv(
    "CONTACT_EMAIL",
    "B40077@students.ucu.ac.ug"
)

FORECAST_DAYS = 7

USER_AGENT = f"UniWeatherAssignment/1.0 ({CONTACT_EMAIL})"

GEOCODE_URL = "https://geocoding-api.open-meteo.com/v1/search"
FORECAST_URL = "https://api.open-meteo.com/v1/forecast"

# PART 1: GEOCODING USING urllib

print("\n--- Geocoding with urllib ---")

params = {
    "name": LOCATION,
    "count": 5,
    "language": "en",
    "format": "json"
}

url = GEOCODE_URL + "?" + urllib.parse.urlencode(params)

req = urllib.request.Request(
    url,
    headers={"User-Agent": USER_AGENT}
)

t0 = time.perf_counter()

with urllib.request.urlopen(req, timeout=10) as response:
    status = response.status
    content_type = response.headers.get("Content-Type")
    raw = response.read()

elapsed = time.perf_counter() - t0

data_urllib = json.loads(raw.decode("utf-8"))

print("Status:", status)
print("Content-Type:", content_type)
print("Time:", f"{elapsed:.3f}s")
print("Results found:", len(data_urllib.get("results", [])))

# PART 2: GEOCODING USING requests
print("\n--- Geocoding with requests ---")

t0 = time.perf_counter()

r = requests.get(
    GEOCODE_URL,
    params=params,
    headers={"User-Agent": USER_AGENT},
    timeout=10
)

elapsed_requests = time.perf_counter() - t0

r.raise_for_status()

data_requests = r.json()

print("Status:", r.status_code)
print("Content-Type:", r.headers.get("Content-Type"))
print("Time:", f"{elapsed_requests:.3f}s")
print(
    "Same data from both approaches:",
    data_requests == data_urllib
)

# CHOOSE LOCATION
results = data_requests.get("results", [])

if not results:
    print("No matching location found.")
    exit()

location = results[0]

lat = location["latitude"]
lon = location["longitude"]
name = location["name"]
country = location.get("country", "")
timezone = location.get("timezone", "auto")

print("\nSelected Location:")
print(f"{name}, {country}")
print("Latitude:", lat)
print("Longitude:", lon)

# FORECAST REQUEST

forecast_params = {
    "latitude": lat,
    "longitude": lon,
    "daily": (
        "weather_code,"
        "temperature_2m_max,"
        "temperature_2m_min,"
        "precipitation_sum,"
        "wind_speed_10m_max"
    ),
    "forecast_days": FORECAST_DAYS,
    "timezone": timezone
}

print("\n--- Retrieving Forecast ---")

forecast_response = requests.get(
    FORECAST_URL,
    params=forecast_params,
    headers={"User-Agent": USER_AGENT},
    timeout=10
)

forecast_response.raise_for_status()

forecast = forecast_response.json()

daily = forecast["daily"]

# WEATHER CODE TRANSLATION

WMO_CODES = {
    0: "Clear Sky",
    1: "Mainly Clear",
    2: "Partly Cloudy",
    3: "Overcast",
    45: "Fog",
    48: "Rime Fog",
    51: "Light Drizzle",
    53: "Moderate Drizzle",
    55: "Dense Drizzle",
    61: "Slight Rain",
    63: "Moderate Rain",
    65: "Heavy Rain",
    71: "Slight Snow",
    73: "Moderate Snow",
    75: "Heavy Snow",
    80: "Rain Showers",
    81: "Moderate Showers",
    82: "Violent Showers",
    95: "Thunderstorm",
    96: "Thunderstorm with Hail",
    99: "Severe Thunderstorm with Hail",
}

# DISPLAY FORECAST

print(f"\n{FORECAST_DAYS}-Day Forecast for {name}, {country}")
print("-" * 70)

for i in range(len(daily["time"])):

    date = daily["time"][i]

    weather_code = daily["weather_code"][i]

    condition = WMO_CODES.get(
        weather_code,
        f"Unknown ({weather_code})"
    )

    max_temp = daily["temperature_2m_max"][i]
    min_temp = daily["temperature_2m_min"][i]
    rain = daily["precipitation_sum"][i]
    wind = daily["wind_speed_10m_max"][i]

    print(f"Date: {date}")
    print(f"Condition: {condition}")
    print(f"Max Temp: {max_temp} °C")
    print(f"Min Temp: {min_temp} °C")
    print(f"Rainfall: {rain} mm")
    print(f"Wind Speed: {wind} km/h")
    print("-" * 70)

# SUMMARY
warmest_index = daily["temperature_2m_max"].index(
    max(daily["temperature_2m_max"])
)

coldest_index = daily["temperature_2m_min"].index(
    min(daily["temperature_2m_min"])
)

total_rain = sum(daily["precipitation_sum"])

print("\nSUMMARY")
print("=" * 70)

print(
    f"Warmest Day: "
    f"{daily['time'][warmest_index]} "
    f"({daily['temperature_2m_max'][warmest_index]} °C)"
)

print(
    f"Coldest Day: "
    f"{daily['time'][coldest_index]} "
    f"({daily['temperature_2m_min'][coldest_index]} °C)"
)

print(f"Total Rainfall: {total_rain:.1f} mm")

print("\nData Source: Open-Meteo API")

#Flow
"""The weather forecasting application follows a straight forward sequence of steps from receiving the user's location to displaying the forecast.
It accepts a location name i.e Mpigi for this case and sends it to the Open_Meteo Geocoding API. These coordinates are then used to request weather forecast data from the Open_meteo Forecast API."""
"""
+------------------+
| User Input       |
| "Mpigi"          |
+--------+---------+
         |
         v
+------------------+
| Geocoding API    |
| Open-Meteo       |
+--------+---------+
         |
         | Returns latitude,
         | longitude, timezone
         v
+------------------+
| Forecast API     |
| Open-Meteo       |
+--------+---------+
         |
         | Returns weather
         | forecast JSON
         v
+------------------+
| Data Processing  |
| Python Script    |
+--------+---------+
         |
         v
+------------------+
| Display Forecast |
| to User          |
+------------------+
"""
"""The application demonstrates 2 methods of communicating with web services i.e. urllib from the python's standard library and request which is an external third-party library.
These retrieve the same geocoding data, allowing comparisons of ease of utilization, response handling and code complexity"""


#Licensing and usage terms of each API, and what they mean for a real product;

"""Open_Meteo API. The project uses the Open-Meteo Geocoding API and Forecast API.
Open-Meteo provides its weather data under the Creative Commons Attribution 4.0 International (CC BY 4.0) license.
This license allows developers to copy, redistribute, modify, and build upon the data, including for commercial purposes, provided appropriate attribution is given to Open-Meteo.
For a real product, the requirements imply;
    1. The application must credit Open-Meteo as the source of weather data.
    2. Developers should monitor API usage to avoid exceeding free tier limits.
    3. Commercial projects may need a paid subscription.
    4. A link to Open-Meteo should be displayed where the forecast information is shown."""
"""Open-Meteo Geocoding Data.
The geocoding service used by Open-Meteo relies partly on GeoNames data.
GeoNames itself requires attribution under its licensing conditions.
Therefore, commercial software should ensure that attribution requirements are satisfied for both Open-Meteo and the underlying location data."""

"""MET Norway API
Unlike many weather services, MET Norway permits commercial use without charging licensing fees.
However, several operational requirements apply:
Requests must include an identifying User-Agent.
The User-Agent should include application information and contact details.
Excessive or abusive traffic may result in blocking.
There is no guaranteed service level agreement (SLA)."""

#reliability issues you met, and how your code handles them;
"""Weather applications depend on external services and internet connectivity. 
Several reliability issues can occur during operation. These include; 
Invalid Location names-user may enter a location that does not exist
Network connectivity failure_possible causes include loss of internet access, DNS failures, API outages and firewall restrictions. In this the script timeouts.
Slow server responses_weather APIs may occasionally respond slowly due to heavy deman. Without a timeout, the script could remain stuck waiting for data.The timeout limits waiting time and makes the application more responsive.
HTTP Errors_ Servers may return error responses. This has been handled where with r.raise_for_status() a failed request is immediately reported rather than accepting invalid data to continue in the work flow."""

#privacy implications of sending location queries to third parties;
"""Though weather forecasts seem harmless, they involve sharing location-related information with external providers.
Disclosure of location: When a user searches for a place, the API provider learns: location entered, request time, surce IP address and this information may reveal; home locations, workplaces, travel plans and frequently visited places.
User-Agent Information: The application sends; USER_AGENT = "UniWeatherAssignment/1.0 (email)" which includes a contact email address.
Server Logging_API providers commonly store IP address, request timestamps, requested locations and browser or application information. such data may be retained for monitoring, analytics, troubleshooting or abuse prevention.
Data minimisation_A key privacy principle is to send information that is necessary. Practical privacy measures include; sending only city names when possible, avoiding transimission of precise personal coordinates, caching previously retrieved forecasts amd avoiding unnecessary repeat request.
Legal consideration_For a production system deployed in Uganda, privacy requirements would need to comply with the Data Protection and Privacy Act, 2019, together with any applicable international regulations. Users should be informed that forecast requests are processed by third-party weather providers and that location information may be transmitted outside the country."""


--- Geocoding with urllib ---
Status: 200
Content-Type: application/json; charset=utf-8
Time: 0.961s
Results found: 3

--- Geocoding with requests ---
Status: 200
Content-Type: application/json; charset=utf-8
Time: 0.907s
Same data from both approaches: False

Selected Location:
Mpigi, Uganda
Latitude: 0.225
Longitude: 32.31361

--- Retrieving Forecast ---

7-Day Forecast for Mpigi, Uganda
----------------------------------------------------------------------
Date: 2026-10-15
Condition: Moderate Showers
Max Temp: 26.7 °C
Min Temp: 19.5 °C
Rainfall: 7.3 mm
Wind Speed: 15.0 km/h
----------------------------------------------------------------------
Date: 2026-10-16
Condition: Violent Showers
Max Temp: 25.2 °C
Min Temp: 18.8 °C
Rainfall: 15.8 mm
Wind Speed: 15.8 km/h
----------------------------------------------------------------------
Date: 2026-10-17
Condition: Moderate Showers
Max Temp: 24.2 °C
Min Temp: 17.9 °C
Rainfall: 19.0 mm
Wind Speed: 8.0 km/h
---------------------------------

'Though weather forecasts seem harmless, they involve sharing location-related information with external providers.\nDisclosure of location: When a user searches for a place, the API provider learns: location entered, request time, surce IP address and this information may reveal; home locations, workplaces, travel plans and frequently visited places.\nUser-Agent Information: The application sends; USER_AGENT = "UniWeatherAssignment/1.0 (email)" which includes a contact email address.\nServer Logging_API providers commonly store IP address, request timestamps, requested locations and browser or application information. such data may be retained for monitoring, analytics, troubleshooting or abuse prevention.\nData minimisation_A key privacy principle is to send information that is necessary. Practical privacy measures include; sending only city names when possible, avoiding transimission of precise personal coordinates, caching previously retrieved forecasts amd avoiding unnecessary rep